In [7]:
# import json

# with open("../../dictionary/output/wiktionary_ainu_glossed_morphemes.json", "r") as f:
#     data = json.load(f)
#     print("morphemes       ", list(data.keys()) [:30])

# with open("../output/ainu_words_all.tsv", "r") as f:
#     words = [line.split("\t", 1)[0] for line in f.read().splitlines()]
#     print("words           ", words[:30])

# # POS for unbound morphemes
# with open("../../dictionary/output/wiktionary_ainu_part_of_speech.json", "r") as f:
#     part_of_speech = json.load(f)
#     print("part_of_speech  ", list(part_of_speech.keys())[:30])

# with open("../../dictionary/output/ainu-itah/sakhalin-terms-extended.json", "r") as f:
#     karahuto_terms = json.load(f)
#     karahuto_terms_translated = [
#         term["lemma"] for term in karahuto_terms if "ja" in term and term["ja"]
#     ]
#     print("karahuto_terms  ", karahuto_terms_translated[:30])




In [8]:
with open("../output/ainu_words_all.tsv", "r") as f:
    words = [line.split("\t", 1)[0] for line in f.read().splitlines()]
    print("words_in_corpora", words[:30])

words_in_corpora ['a=', 'ne', 'wa', '=an', 'an', 'kor', 'ka', 'ta', 'ki', 'e=', 'kusu', 'ruwe', 'pe', 'an=', "'", 'i=', 'hine', 'kamuy', 'p', 'oka', 'kane', 'a', 'na', 'taa', 'ku=', 'or', 'utar', 'hi', 'sekor', 'ye']


In [9]:
import json
with open("../../dictionary/output/combined_glosses.json", "r") as f:
    combined_glosses = json.load(f)
    print("combined glosses", list(combined_glosses.keys())[:30])

with open("../../dictionary/output/combined_part_of_speech.json", "r") as f:
    combined_pos = json.load(f)
    print("combined part of speech", list(combined_pos.keys())[:30])

with open("../../dictionary/output/combined_word_compositions.json", "r") as f:
    combined_word_compositions = json.load(f)
    print("combined word compositions", list(combined_word_compositions.keys())[:30])

combined glosses ['wan', 'tu', 'rak', 'mi', 'on', 'ona', 'ay', 'oro', 'he', 'i', 'ne', 'ni', 're', 'as', 'te', 'no', 'os', 'or', 'pet', 'an', 'am', 'us', 'ani', 'mun', 'her', 'n', 'p', 'not', 'sine', 'hi']
combined part of speech ['wan', 'tu', 'rak', 'ci', 'mi', 'on', 'ona', 'o', 'ay', 'oro', 'he', 'i', 'ne', 'si', 'ni', 're', 'as', 'te', 'no', 'un', 'os', 'or', 'pet', 'e', 'mon', 'an', 'am', 'us', 'a', 'ani']
combined word compositions ['rep', 'tanpa', 'oyapa', 'ahupte', 'ahunke', 'ari', 'anpe', 'hawean', 'ikure', 'ikuruy', 'isepo', 'ipere', 'iyayiraykere', 'irara', 'ipepasuy', 'ikupasuy', 'apepasuy', 'icakkerere', 'imeru', 'iruska', 'upaskuma', 'ikonnupkoan', 'sisam', 'ekte', 'hetuku', 'siroma', 'sikkap', 'sikkes', 'nukar', 'itakno']


In [10]:
freq_map: dict[str, int] = {}

with open("../output/ainu_words_all.tsv", "r") as f:
    for line in f.read().splitlines():
        word, freq = line.split("\t", 1)
        freq_map[word] = int(freq)

In [11]:
import regex as re


def can_segment(word, known_morphemes):
    """
    Return True if 'word' can be segmented entirely by
    the set/list of known morphemes.
    """
    dp = [False] * (len(word) + 1)
    dp[0] = True
    for i in range(len(word)):
        if dp[i]:
            for morpheme in known_morphemes:
                if word.startswith(morpheme, i):
                    dp[i + len(morpheme)] = True
    return dp[-1]


covered = []
uncovered = []

for w in words:
    # Strip out boundary markers
    w_normalized = re.sub(r"[=\-]+", "", w)

    # If the normalized form appears in 'part_of_speech', count it as covered
    # if w_normalized in part_of_speech or w_normalized in karahuto_terms_translated:
    #     covered.append(w)
    # else:
    #     # Otherwise, try to segment it with known morphemes
    #     if can_segment(w_normalized, data.keys()):
    #         covered.append(w)
    #     else:
    #         uncovered.append(w)

    if w_normalized in combined_pos or w_normalized in combined_word_compositions:
        covered.append(w)
    else:
        if can_segment(w_normalized, combined_glosses.keys()):
            covered.append(w)
        else:
            uncovered.append(w)

uncovered_words_with_freq = [w for w in uncovered if freq_map[w] > 1]

print("Number of covered words: ", len(covered))
print("Number of uncovered words:", len(uncovered))
print(
    "Number of uncovered words with frequency > 1: ",
    len(uncovered_words_with_freq),
)

Number of covered words:  23522
Number of uncovered words: 12337
Number of uncovered words with frequency > 1:  4980


In [12]:
# covered words * freq / total words * 100
print("Coverage: ", sum(freq_map[w] for w in covered) / sum(freq_map[w] for w in words) * 100)



Coverage:  92.04973608380801


In [13]:
# # Cumulative coverage
# cumulative_coverage = 0
# for w in sorted(freq_map, key=freq_map.get, reverse=True):
#     cumulative_coverage += freq_map[w]
#     print(f"{w}: {cumulative_coverage / sum(freq_map.values()) * 100:.2f}%")

In [14]:
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

def create_coverage_visualizations(
    freq_map, covered, uncovered, output_dir="output/figures"
):
    # Create output directory if it doesn't exist
    Path(output_dir).mkdir(parents=True, exist_ok=True)

    # 1. Frequency Distribution Plot
    plt.figure(figsize=(12, 6))
    # Convert frequencies to DataFrame
    frequencies_df = pd.DataFrame({"frequency": list(freq_map.values())})
    frequencies_df["log_frequency"] = np.log10(frequencies_df["frequency"])

    sns.histplot(data=frequencies_df, x="log_frequency", bins=50)
    plt.title("Word Frequency Distribution (Log Scale)")
    plt.xlabel("Log10(Frequency)")
    plt.ylabel("Count")
    plt.savefig(
        f"{output_dir}/frequency_distribution.png", dpi=300, bbox_inches="tight"
    )
    plt.close()

    # 2. Coverage vs Frequency Threshold
    thresholds = range(1, 21)
    coverage_data = []
    for threshold in thresholds:
        words_above_threshold = {w: f for w, f in freq_map.items() if f >= threshold}
        total_tokens = sum(words_above_threshold.values())
        covered_tokens = sum(words_above_threshold.get(w, 0) for w in covered)
        coverage_rate = (covered_tokens / total_tokens) * 100 if total_tokens > 0 else 0
        coverage_data.append({"threshold": threshold, "coverage_rate": coverage_rate})

    coverage_df = pd.DataFrame(coverage_data)
    plt.figure(figsize=(12, 6))
    sns.lineplot(data=coverage_df, x="threshold", y="coverage_rate", marker="o")
    plt.title("Coverage Rate vs Frequency Threshold")
    plt.xlabel("Minimum Frequency Threshold")
    plt.ylabel("Coverage Rate (%)")
    plt.grid(True)
    plt.savefig(f"{output_dir}/coverage_by_threshold.png", dpi=300, bbox_inches="tight")
    plt.close()

    # 3. Top Uncovered Words
    plt.figure(figsize=(15, 8))
    uncovered_freq = {w: freq_map[w] for w in uncovered}
    top_uncovered = dict(
        sorted(uncovered_freq.items(), key=lambda x: x[1], reverse=True)[:20]
    )

    top_uncovered_df = pd.DataFrame(
        {"word": list(top_uncovered.keys()), "frequency": list(top_uncovered.values())}
    )

    sns.barplot(data=top_uncovered_df, x="word", y="frequency")
    plt.xticks(rotation=45, ha="right")
    plt.title("Top 20 Most Frequent Uncovered Words")
    plt.xlabel("Word")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.savefig(f"{output_dir}/top_uncovered_words.png", dpi=300, bbox_inches="tight")
    plt.close()

    # 4. Coverage Summary
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

    # Types (unique words) coverage
    types_data = pd.DataFrame(
        {"category": ["Covered", "Uncovered"], "count": [len(covered), len(uncovered)]}
    )

    # Tokens (total occurrences) coverage
    tokens_covered = sum(freq_map[w] for w in covered)
    tokens_uncovered = sum(freq_map[w] for w in uncovered)
    tokens_data = pd.DataFrame(
        {
            "category": ["Covered", "Uncovered"],
            "count": [tokens_covered, tokens_uncovered],
        }
    )

    # Plot pie charts
    ax1.pie(types_data["count"], labels=types_data["category"], autopct="%1.1f%%")
    ax1.set_title("Coverage by Types (Unique Words)")

    ax2.pie(tokens_data["count"], labels=tokens_data["category"], autopct="%1.1f%%")
    ax2.set_title("Coverage by Tokens (Total Occurrences)")

    plt.savefig(f"{output_dir}/coverage_summary.png", dpi=300, bbox_inches="tight")
    plt.close()

    # Print summary statistics
    print("Coverage Statistics:")
    print(f"Total unique words: {len(freq_map)}")
    print(f"Covered words: {len(covered)} ({len(covered)/len(freq_map)*100:.1f}%)")
    print(f"Total tokens: {sum(freq_map.values())}")
    print(
        f"Covered tokens: {tokens_covered} ({tokens_covered/sum(freq_map.values())*100:.1f}%)"
    )

    # Create detailed CSV report
    df = pd.DataFrame(
        {
            "word": list(freq_map.keys()),
            "frequency": list(freq_map.values()),
            "is_covered": [w in covered for w in freq_map.keys()],
        }
    )
    df.sort_values("frequency", ascending=False).to_csv(
        f"{output_dir}/word_coverage_report.csv",
        sep="\t",
        index=False,
        encoding="utf-8",
    )

create_coverage_visualizations(freq_map, covered, uncovered, output_dir="output/figures")

Coverage Statistics:
Total unique words: 35859
Covered words: 23522 (65.6%)
Total tokens: 1153207
Covered tokens: 1061524 (92.0%)


In [15]:
for word in sorted(uncovered_words_with_freq, key=lambda x: freq_map[x], reverse=True):
    print(f"{word}", freq_map[word])

' 11632
k= 2887
or_ 1782
__hine 1766
u 1733
V 1654
_wa 1515
_hine 1511
__hi 1366
teh 1355
kor_ 1270
_un 1116
nah 1092
ne'ampe 918
_hi 847
monimahpo 810
_hike 631
_a 611
an_= 562
’ 531
cis' 496
__hike 484
kes 447
w_a 437
h_i 435
_akusu 427
h_ine 421
inkar' 410
_akus 398
neeteh 317
néno 311
haw'as 297
okayahci 291
utah 271
acahcipo 269
_hene 263
A 252
an_ 245
nankor_ 234
Ne 234
_hetap 233
koh 222
tah 208
c= 198
haw'an 197
híne 197
_ayne 195
ne'an 195
sap' 192
'iine'ahsuy 189
Otasam 188
B 184
yuhpo 180
wen'iruska 175
húci 174
Ku= 172
sísam 170
sir'an 161
kihci 156
néwaanpe 151
_an 145
ar 145
harkakkok 141
néa 139
nuyé 139
nukar_ 133
utar_ 132
Ipet 131
sapahci 129
_a= 121
M 120
_ene 116
_an= 114
síno 113
sineh 113
_yakanak 111
ahup' 110
yahka 110
rittunna 108
Tane 108
ahkapo 107
sik'o 106
Sinutapka 105
heururu 104
Kayano 102
káni 101
wensirkur'ante 100
yehci 100
Poyyaunpe 98
=an_ 96
ehci 96
kotom'an 95
'e 95
cip'o 95
siwto 95
umma 95
etuhka 95
wahka 94
_or 93
nukár 93
Korka 93
kar_ 92
póka

In [16]:
def create_cumulative_coverage_plot(freq_map, covered, output_dir="output/figures"):
    # Sort words by frequency
    sorted_words = sorted(freq_map.items(), key=lambda x: x[1], reverse=True)
    total_tokens = sum(freq_map.values())

    # Calculate cumulative coverage
    cumulative_data = []
    cumulative_tokens = 0
    covered_tokens = 0

    for rank, (word, freq) in enumerate(sorted_words, 1):
        cumulative_tokens += freq
        if word in covered:
            covered_tokens += freq

        cumulative_data.append(
            {
                "rank": rank,
                "cumulative_coverage": covered_tokens / total_tokens * 100,
                "total_coverage": cumulative_tokens / total_tokens * 100,
            }
        )

    # Create DataFrame
    df = pd.DataFrame(cumulative_data)

    # Create plot
    plt.figure(figsize=(12, 6))

    # Plot cumulative coverage
    sns.lineplot(
        data=df, x="rank", y="cumulative_coverage", label="Covered Words", color="blue"
    )

    # Plot total possible coverage as dotted red line
    sns.lineplot(
        data=df,
        x="rank",
        y="total_coverage",
        label="Total Words",
        color="red",
        linestyle="--",
    )

    plt.xscale("log")  # Use log scale for rank
    plt.grid(True, which="both", ls="-", alpha=0.2)
    plt.title("Cumulative Word Coverage by Rank")
    plt.xlabel("Word Rank (log scale)")
    plt.ylabel("Coverage (%)")
    plt.legend()

    plt.tight_layout()
    plt.savefig(f"{output_dir}/cumulative_coverage.png", dpi=300, bbox_inches="tight")
    plt.close()

create_cumulative_coverage_plot(freq_map, covered, output_dir="output/figures")